# Praktikum 6 : pencarian kafe terdekat

In [1]:
# Memuat dan menstandardisasi fitur numerik PBL
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
df = pd.read_csv('../../dataset_pbl/cafes.csv')
features = ['latitude', 'longitude', 'rating', 'review_count', 'price_min', 'price_max']
X = df[features].fillna(df[features].mean())
X_scaled = StandardScaler().fit_transform(X).astype('float32')
k = 10

In [2]:
# Exact Nearest Neighbor
start = time.perf_counter()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.perf_counter() - start
print('Exact NN:', time_exact, 'detik')

Exact NN: 1.7694663999973272 detik


In [3]:
# Annoy
start = time.perf_counter()
index_annoy = AnnoyIndex(X_scaled.shape[1], 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.perf_counter() - start
print('Annoy:', time_annoy, 'detik')

Annoy: 0.008458700001938269 detik


In [4]:
# HNSW
start = time.perf_counter()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=len(X_scaled), ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.perf_counter() - start
print('HNSW:', time_hnsw, 'detik')

HNSW: 0.009898699998302618 detik


In [5]:
# FAISS IVF
start = time.perf_counter()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.perf_counter() - start
print('FAISS IVF:', time_faiss, 'detik')

FAISS IVF: 0.004810500002349727 detik


In [6]:
# Top-5 tetangga dari kafe pertama, termasuk item query
print('Kafe pertama:', df.loc[0, 'name'])
print('Exact NN:', idx_exact[0][:5])
print('Annoy:', idx_annoy[0][:5])
print('HNSW:', idx_hnsw[0][:5])
print('FAISS:', idx_faiss[0][:5])

Kafe pertama: Warkop Djayantie
Exact NN: [ 0 85 40 24 18]
Annoy: [0, 85, 40, 24, 18]
HNSW: [ 0 85 40 24 18]
FAISS: [ 0 85 40 24 18]
